# Validate dual-polarized PMI codebook and right-half muting

This notebook validates the recovered codebook/muting/PMI-mask stack:

- `ArrayConfig` as the single source of truth;
- Dual-polarized subarray DFT codebook (`dft.py`);
- Subarray physical expansion;
- Right-half physical-port muting (`muting.py`);
- Cross-polarization i2 co-phasing;
- PMI ↔ flat beam index conversion (`pmi.py`);
- Normal-peak-direction total array-factor loss (`pmi_mask.py`).

Scope:
- RI = 1;
- Logical subarray grid 4×8, 2 elements per subarray, physical grid 8×8;
- Codebook ordering is `(i12, i11, i2)`, with `i2` fastest;
- Project port ordering is `polarization-major`;
- `valid_pmi_mask[i12, i11]` selects spatial beams;
- Total power = incoherent sum over the two polarizations;
- Loss is evaluated at the normal-state peak direction;
- No element pattern, no Sionna RT scene, no UE/channel/SINR.

In [ ]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch

matplotlib.use("module://matplotlib_inline.backend_inline")

# Allow the notebook to run from either the project root or notebook/ directory.
module_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src" / "mMIMO_sleep" / "codebook" / "dft.py").exists()),
    None,
)
if module_root is None:
    raise FileNotFoundError("Cannot find project root with src/mMIMO_sleep/codebook.")
sys.path.insert(0, str(module_root / "src"))

from mMIMO_sleep.array_config import ArrayConfig
from mMIMO_sleep.codebook.dft import generate_dft_codebook
from mMIMO_sleep.codebook.muting import (
    active_power_fraction,
    apply_muting_mask,
    create_right_half_mask,
)
from mMIMO_sleep.codebook.pmi import PMI, beam_index_to_pmi, pmi_to_beam_index
from mMIMO_sleep.codebook.pmi_mask import (
    IDEAL_HALF_APERTURE_LOSS_DB,
    beam_indices_from_mask,
    compute_total_peak_loss_db,
    create_total_loss_pmi_mask,
    pmi_indices_from_mask,
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 1. ArrayConfig and DFT codebook

In [ ]:
# Project's standard subarray configuration.
config = ArrayConfig(
    num_subarray_rows=4,
    num_horizontal=8,
    elements_per_subarray=2,
    num_polarizations=2,
)

NUM_VERTICAL_BEAMS = 8
NUM_HORIZONTAL_BEAMS = 32
NUM_I2 = 4
PHASE_SIGN = 1
SPACING_WAVELENGTHS = 0.5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("num_subarray_rows:", config.num_subarray_rows)
print("num_horizontal (columns):", config.num_horizontal)
print("elements_per_subarray:", config.elements_per_subarray)
print("num_polarizations:", config.num_polarizations)
print("num_physical_rows:", config.num_physical_rows)
print("num_physical_elements:", config.num_physical_elements)
print("num_physical_ports:", config.num_physical_ports)

codebook = generate_dft_codebook(
    config,
    num_vertical_beams=NUM_VERTICAL_BEAMS,
    num_horizontal_beams=NUM_HORIZONTAL_BEAMS,
    num_i2=NUM_I2,
    phase_sign=PHASE_SIGN,
    dtype=torch.complex64,
    device=DEVICE,
)

print("codebook shape:", tuple(codebook.shape))
# Expected: (NUM_VERTICAL_BEAMS * NUM_HORIZONTAL_BEAMS * NUM_I2, config.num_physical_ports)
expected_shape = (NUM_VERTICAL_BEAMS * NUM_HORIZONTAL_BEAMS * NUM_I2, config.num_physical_ports)
assert codebook.shape == expected_shape

# Each codeword is unit norm over all physical ports.
norms = codebook.abs().square().sum(dim=-1)
torch.testing.assert_close(norms, torch.ones_like(norms), atol=1e-5, rtol=1e-5)

# Polarization-major ordering: first half of ports are pol0, second half pol1.
num_elements = config.num_physical_elements
pol0 = codebook[0, :num_elements]
pol1 = codebook[0, num_elements:]
# For i2=0, pol0 and pol1 should be identical.
ratio = pol1 / (pol0 + 1e-12)
angle_diff = (torch.angle(ratio) - 0.0 + math.pi) % (2 * math.pi) - math.pi
assert torch.all(angle_diff.abs() < 1e-4), "pol0/pol1 should be co-phased for i2=0"

print("Codebook shape/norm check: PASS")

## 2. Subarray physical expansion

In [ ]:
# Pick a spatial beam with non-trivial vertical phase progression
# so adjacent logical subarray rows differ.
sub_spatial_index = pmi_to_beam_index(
    PMI(i11=5, i12=3, i2=0),
    num_horizontal_beams=NUM_HORIZONTAL_BEAMS,
    num_vertical_beams=NUM_VERTICAL_BEAMS,
    num_i2=NUM_I2,
) // NUM_I2

w = codebook[sub_spatial_index * NUM_I2]
pol0 = w[:num_elements].reshape(config.num_physical_rows, config.num_horizontal)
pol1 = w[num_elements:].reshape(config.num_physical_rows, config.num_horizontal)
eps = config.elements_per_subarray

for pol_name, pol_w in [("pol0", pol0), ("pol1", pol1)]:
    for logical_row in range(config.num_subarray_rows):
        start = logical_row * eps
        end = start + eps
        subarray_rows = pol_w[start:end, :]
        # All physical rows belonging to the same logical subarray row
        # must share the same spatial weight.
        torch.testing.assert_close(
            subarray_rows,
            subarray_rows[0:1, :].expand_as(subarray_rows),
            atol=1e-5,
            rtol=1e-5,
        )
    # Because the chosen vertical beam index is not 0, different logical
    # subarray rows should not all be identical.
    for r0 in range(config.num_subarray_rows - 1):
        assert not torch.allclose(
            pol_w[r0 * eps:(r0 + 1) * eps, :],
            pol_w[(r0 + 1) * eps:(r0 + 2) * eps, :],
            atol=1e-4,
        ), "Adjacent logical subarray rows unexpectedly identical"

print("Subarray expansion: PASS")

## 3. i2 ordering and co-phasing

In [ ]:
# Fix a spatial beam and sweep i2.
spatial_index = (
    pmi_to_beam_index(
        PMI(i11=7, i12=2, i2=0),
        num_horizontal_beams=NUM_HORIZONTAL_BEAMS,
        num_vertical_beams=NUM_VERTICAL_BEAMS,
        num_i2=NUM_I2,
    )
    // NUM_I2
)

flat_indices = []
for i2 in range(NUM_I2):
    idx = spatial_index * NUM_I2 + i2
    flat_indices.append(idx)
    pol0 = codebook[idx, :num_elements]
    pol1 = codebook[idx, num_elements:]

    # pol0 must be identical across all i2 values.
    torch.testing.assert_close(pol0, codebook[spatial_index * NUM_I2, :num_elements], atol=1e-6, rtol=1e-6)

    expected_phase = torch.exp(
        torch.tensor(
            1j * 2.0 * math.pi * i2 / NUM_I2,
            device=DEVICE,
            dtype=torch.complex64,
        )
    )
    torch.testing.assert_close(
        pol1,
        pol0 * expected_phase,
        atol=1e-5,
        rtol=1e-5,
    )

    # Each codeword remains unit norm.
    norm = codebook[idx].abs().square().sum()
    torch.testing.assert_close(norm, torch.tensor(1.0, device=DEVICE), atol=1e-5, rtol=1e-5)

# The four flat indices for this spatial PMI must be consecutive.
assert flat_indices == list(range(flat_indices[0], flat_indices[0] + NUM_I2))
print("i2 ordering/co-phasing: PASS")

## 4. Right-half muting

In [ ]:
muting_mask = create_right_half_mask(config, dtype=torch.float32, device=DEVICE)
sleep_codebook = apply_muting_mask(codebook, muting_mask)

print("muting_mask shape:", tuple(muting_mask.shape))
print("sleep_codebook shape:", tuple(sleep_codebook.shape))

# Exactly half the horizontal columns are muted, so active power fraction = 0.5.
apf = active_power_fraction(muting_mask)
print("active_power_fraction:", apf)
assert abs(apf - 0.5) < 1e-6

# Sleep codebook is zero on muted ports.
muted_ports = (muting_mask == 0).nonzero(as_tuple=True)[0]
assert torch.allclose(
    sleep_codebook[:, muted_ports],
    torch.zeros_like(sleep_codebook[:, muted_ports]),
    atol=1e-7,
)

# Per-codeword sleep power should be ~0.5 of normal power.
normal_power = codebook.abs().square().sum(dim=-1)
sleep_power = sleep_codebook.abs().square().sum(dim=-1)
tx_power_ratio = sleep_power / normal_power
torch.testing.assert_close(
    tx_power_ratio,
    torch.full_like(tx_power_ratio, 0.5),
    atol=1e-5,
    rtol=1e-5,
)
print("Right-half Muting/no-renormalization: PASS")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
mask_2d = (
    muting_mask[:num_elements]
    .reshape(config.num_physical_rows, config.num_horizontal)
    .cpu()
    .numpy()
)
im = ax.imshow(mask_2d, vmin=0, vmax=1, cmap="Blues", origin="upper")
ax.set_title("Right-half muting mask (pol0)")
ax.set_xlabel("Horizontal element index")
ax.set_ylabel("Vertical element index")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()

## 5. PMI and flat beam index

In [ ]:
pmi = PMI(i11=17, i12=3, i2=1)
beam_idx = pmi_to_beam_index(
    pmi,
    num_horizontal_beams=NUM_HORIZONTAL_BEAMS,
    num_vertical_beams=NUM_VERTICAL_BEAMS,
    num_i2=NUM_I2,
)
pmi_back = beam_index_to_pmi(
    beam_idx,
    num_horizontal_beams=NUM_HORIZONTAL_BEAMS,
    num_vertical_beams=NUM_VERTICAL_BEAMS,
    num_i2=NUM_I2,
)
assert pmi_back == pmi
print(f"PMI {pmi} -> beam index {beam_idx} -> PMI {pmi_back}")

# Beam ordering: (i12, i11, i2) with i2 fastest.
expected = (pmi.i12 * NUM_HORIZONTAL_BEAMS + pmi.i11) * NUM_I2 + pmi.i2
assert beam_idx == expected
print("PMI/Beam index round-trip: PASS")

## 6. Normal-peak-direction total loss and validity mask

In [ ]:
# Compute total peak loss for every spatial PMI.
# i2 invariance means only one i2 block is used internally.
loss_db = compute_total_peak_loss_db(
    config,
    num_vertical_beams=NUM_VERTICAL_BEAMS,
    num_horizontal_beams=NUM_HORIZONTAL_BEAMS,
    num_i2=NUM_I2,
    phase_sign=PHASE_SIGN,
    spacing_wavelengths=SPACING_WAVELENGTHS,
    grid_points=121,
    beam_batch_size=32,
    dtype=torch.complex64,
    device=DEVICE,
)

assert loss_db.shape == (NUM_VERTICAL_BEAMS, NUM_HORIZONTAL_BEAMS)
assert loss_db.dtype == torch.float32
print("Loss shape:", tuple(loss_db.shape))
print("Loss median:", float(loss_db.median().item()), "dB")
print("Loss finite count:", int(torch.isfinite(loss_db).sum().item()))

In [ ]:
# Beams whose loss is within 1 dB of the ideal half-aperture loss are valid.
valid_mask = create_total_loss_pmi_mask(
    config,
    num_vertical_beams=NUM_VERTICAL_BEAMS,
    num_horizontal_beams=NUM_HORIZONTAL_BEAMS,
    num_i2=NUM_I2,
    target_loss_db=IDEAL_HALF_APERTURE_LOSS_DB,
    tolerance_db=1.0,
    phase_sign=PHASE_SIGN,
    spacing_wavelengths=SPACING_WAVELENGTHS,
    grid_points=121,
    dtype=torch.complex64,
    device=DEVICE,
)

assert valid_mask.shape == (NUM_VERTICAL_BEAMS, NUM_HORIZONTAL_BEAMS)
assert valid_mask.dtype == torch.bool

fig, ax = plt.subplots(figsize=(10, 3))
im = ax.imshow(
    valid_mask.cpu().numpy(),
    origin="lower",
    aspect="auto",
    cmap="Greens",
)
ax.set_xlabel("i11 (horizontal PMI index)")
ax.set_ylabel("i12 (vertical PMI index)")
ax.set_title(
    "Valid spatial PMI mask (target=%.2f dB, tol=\u00b11.0 dB)"
    % IDEAL_HALF_APERTURE_LOSS_DB
)
ax.set_xticks(range(0, NUM_HORIZONTAL_BEAMS, 4))
ax.set_yticks(range(NUM_VERTICAL_BEAMS))
fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()

print("Valid spatial PMIs:", int(valid_mask.sum().item()))

In [ ]:
# Loss heatmap: x-axis i11, y-axis i12, origin lower so i12=0 at bottom.
fig, ax = plt.subplots(figsize=(10, 3.5))
im = ax.imshow(
    loss_db.cpu().numpy(),
    origin="lower",
    aspect="auto",
    cmap="viridis",
)
ax.set_xlabel("i11 (horizontal PMI index)")
ax.set_ylabel("i12 (vertical PMI index)")
ax.set_title("Normal-peak-direction EIRP loss (dB)")
ax.set_xticks(range(0, NUM_HORIZONTAL_BEAMS, 4))
ax.set_yticks(range(NUM_VERTICAL_BEAMS))
fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()

In [ ]:
# Histogram of finite loss values; inf/nan entries are reported separately.
finite_loss = loss_db[torch.isfinite(loss_db)].cpu().numpy()
inf_count = int((~torch.isfinite(loss_db)).sum().item())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(finite_loss, bins=50, color="steelblue", edgecolor="white")
ax.axvline(IDEAL_HALF_APERTURE_LOSS_DB, color="red", linestyle="-", label="target")
ax.axvline(
    IDEAL_HALF_APERTURE_LOSS_DB - 1.0,
    color="green",
    linestyle="--",
    label="target - tol",
)
ax.axvline(
    IDEAL_HALF_APERTURE_LOSS_DB + 1.0,
    color="green",
    linestyle="--",
    label="target + tol",
)
ax.set_xlabel("Loss (dB)")
ax.set_ylabel("Count")
ax.set_title("Distribution of finite normal-peak-direction losses")
ax.legend()
plt.show()

print("Finite loss count:", len(finite_loss))
print("Inf/NaN loss count:", inf_count)

## 7. Representative valid and invalid PMIs

In [ ]:
diff = (loss_db - IDEAL_HALF_APERTURE_LOSS_DB).abs()
best_flat = torch.argmin(diff).item()
i12_best = best_flat // NUM_HORIZONTAL_BEAMS
i11_best = best_flat % NUM_HORIZONTAL_BEAMS
best_loss = loss_db[i12_best, i11_best].item()

def flat_indices_for_pmi(i11: int, i12: int) -> list[int]:
    return [
        pmi_to_beam_index(
            PMI(i11=i11, i12=i12, i2=i2),
            num_horizontal_beams=NUM_HORIZONTAL_BEAMS,
            num_vertical_beams=NUM_VERTICAL_BEAMS,
            num_i2=NUM_I2,
        )
        for i2 in range(NUM_I2)
    ]

print("Representative valid PMI:")
print(f"  i11={i11_best}, i12={i12_best}")
print(f"  loss_db = {best_loss:.3f} dB")
print(f"  valid = {valid_mask[i12_best, i11_best].item()}")
print(f"  all i2 flat indices = {flat_indices_for_pmi(i11_best, i12_best)}")

invalid_mask = ~valid_mask
if invalid_mask.any():
    invalid_indices = invalid_mask.nonzero(as_tuple=False)
    worst_idx_in_invalid = torch.argmax(diff[invalid_mask]).item()
    i12_worst = int(invalid_indices[worst_idx_in_invalid, 0])
    i11_worst = int(invalid_indices[worst_idx_in_invalid, 1])
    worst_loss = loss_db[i12_worst, i11_worst].item()
    print()
    print("Representative invalid PMI (largest deviation from target):")
    print(f"  i11={i11_worst}, i12={i12_worst}")
    print(f"  loss_db = {worst_loss:.3f} dB")
    print(f"  valid = {valid_mask[i12_worst, i11_worst].item()}")
    print(f"  all i2 flat indices = {flat_indices_for_pmi(i11_worst, i12_worst)}")
else:
    print("No invalid PMIs found.")

## 8. Valid PMI expansion to full i2 beams

In [ ]:
spatial_pmi_tuples = pmi_indices_from_mask(valid_mask)
beam_indices = beam_indices_from_mask(valid_mask, num_i2=NUM_I2)

print("Number of valid spatial PMIs:", len(spatial_pmi_tuples))
print("Number of valid full beams (with i2 expansion):", len(beam_indices))

assert len(beam_indices) == NUM_I2 * len(spatial_pmi_tuples)

# Verify a few entries round-trip.
for flat_idx in beam_indices[:4]:
    pmi = beam_index_to_pmi(
        flat_idx,
        num_horizontal_beams=NUM_HORIZONTAL_BEAMS,
        num_vertical_beams=NUM_VERTICAL_BEAMS,
        num_i2=NUM_I2,
    )
    assert valid_mask[pmi.i12, pmi.i11].item()

print("Valid spatial PMI -> all i2 expansion: PASS")

## 9. Optional: inspect one valid PMI pattern

In [ ]:
from mMIMO_sleep.codebook.pmi_mask import (
    _array_factor_from_weights,
    _make_visible_uv_grid,
    _total_power_from_polarized_fields,
)

# Pick a valid spatial PMI near the middle of the valid list.
i11, i12 = spatial_pmi_tuples[len(spatial_pmi_tuples) // 2]
spatial_index = (
    pmi_to_beam_index(
        PMI(i11=i11, i12=i12, i2=0),
        num_horizontal_beams=NUM_HORIZONTAL_BEAMS,
        num_vertical_beams=NUM_VERTICAL_BEAMS,
        num_i2=NUM_I2,
    )
    // NUM_I2
)
w_normal = codebook[spatial_index * NUM_I2]
w_sleep = sleep_codebook[spatial_index * NUM_I2]

u_h, u_v, visible = _make_visible_uv_grid(121, dtype=torch.float32, device=DEVICE)
u_h_f = u_h[visible]
u_v_f = u_v[visible]


def total_power_pattern(weights: torch.Tensor) -> torch.Tensor:
    pol0 = weights[:num_elements].reshape(
        config.num_physical_rows, config.num_horizontal
    )
    pol1 = weights[num_elements:].reshape(
        config.num_physical_rows, config.num_horizontal
    )
    e0 = _array_factor_from_weights(pol0, u_h_f, u_v_f, SPACING_WAVELENGTHS)
    e1 = _array_factor_from_weights(pol1, u_h_f, u_v_f, SPACING_WAVELENGTHS)
    return _total_power_from_polarized_fields(e0, e1)


p_normal = total_power_pattern(w_normal)
p_sleep = total_power_pattern(w_sleep)

p_n_2d = torch.full_like(u_h, float("nan"))
p_s_2d = torch.full_like(u_h, float("nan"))
p_n_2d[visible] = p_normal
p_s_2d[visible] = p_sleep

p_n_db = 10.0 * torch.log10(p_n_2d.clamp_min(1e-12))
p_s_db = 10.0 * torch.log10(p_s_2d.clamp_min(1e-12))

vmax = float(p_n_db[visible].max().item())
vmin = vmax - 35.0

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, data, title in [
    (axes[0], p_n_db, "Normal total power (dB)"),
    (axes[1], p_s_db, "Sleep total power (dB)"),
]:
    im = ax.imshow(
        data.cpu().numpy(),
        origin="lower",
        extent=[-1, 1, -1, 1],
        vmin=vmin,
        vmax=vmax,
        cmap="viridis",
        aspect="equal",
    )
    ax.set_xlabel(r"Horizontal direction cosine $u_h$")
    ax.set_ylabel(r"Vertical direction cosine $u_v$")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.8)
plt.suptitle(f"PMI (i11={i11}, i12={i12}) pattern", y=1.02)
plt.show()

normal_peak_val = p_normal.max().item()
sleep_at_peak = p_sleep[p_normal.argmax()].item()
instant_loss_db = 10.0 * math.log10(normal_peak_val / sleep_at_peak)
print(f"Normal peak power: {normal_peak_val:.6f}")
print(f"Sleep power at normal peak: {sleep_at_peak:.6f}")
print(f"Instantaneous total peak loss: {instant_loss_db:.2f} dB")
print(f"Pre-computed loss[i12={i12}, i11={i11}]: {loss_db[i12, i11].item():.2f} dB")

## 10. Summary

In [ ]:
print("Array configuration: PASS")
print("Codebook shape/norm: PASS")
print("Subarray expansion: PASS")
print("Polarization ordering/power: PASS")
print("i2 ordering/co-phasing: PASS")
print("Right-half Muting/no-renormalization: PASS")
print("Loss/mask axis semantics: PASS")
print("Valid spatial PMI -> all i2 expansion: PASS")